# 第 6 章　面板数据回归
## FE、HDFE、Fama-MacBeth 与 Portfolio Sort

::: {.callout-note}
## 本章要点

1. **辛普森悖论视角**：为什么截面相关会误导因果推断，固定效应如何解决
2. **固定效应的 FWL 理解**：组内去均值 = 对个体虚拟变量做 FWL 残差化
3. **高维固定效应（HDFE）**：`reghdfe` + `pyfixest` 同时控制多维 FE
4. **Fama-MacBeth 回归**：每期截面回归 → 时序均值推断，资产定价标准方法
5. **Portfolio Sort 分组分析**：FM 的非参数对照工具
6. **Python + Stata 双语呈现**

**与第 5 章的连接**：固定效应是 FWL 定理的直接应用——
加入个体虚拟变量并做残差化，等价于组内去均值。
理解了这一点，HDFE 和 DID（第 8 章）就只是「换一种去均值方式」。
:::


## 环境准备


In [ ]:
# ── 第 6 章　面板数据回归　──────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, FamaMacBeth as FM_lm
import pyfixest as pf
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_CLEAN = 'data_clean'
OUTPUT     = 'output'
for d in [DATA_CLEAN, OUTPUT]:
    os.makedirs(d, exist_ok=True)

RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 0　引言：截面相关 ≠ 因果关系

面板数据最大的价值，不是「数据更多」，而是**可以区分两种相关**：

- **截面相关（between variation）**：不同个体之间的差异
- **时序相关（within variation）**：同一个体在不同时期的变化

这两种相关的方向，有时候完全相反。
当它们方向相反时，用截面数据得到的「关系」不仅估计错误，
结论方向可能完全颠倒——这就是**辛普森悖论**在面板数据中的表现形式。

固定效应估计量的本质，就是**只利用时序相关（within variation）**，
丢弃截面相关，从而规避这个问题。


---

## 1　本章演示数据


In [ ]:
# ── 1.1  读入真实数据 or 生成模拟面板 ───────────────────────────────────
try:
    df = pd.read_csv(f'{DATA_CLEAN}/merged_clean.csv', dtype={'stkcd': str})
    need = ['stkcd','year','soe','Size_w','Leverage_w','ROA_w','avg_loan_rate']
    df   = df.dropna(subset=[c for c in need if c in df.columns])
    DATA_SOURCE = 'CSMAR（真实数据）'
    print(f'真实数据：{df.shape}')
except FileNotFoundError:
    # DGP（数据生成过程）——含时不变遗漏变量
    n_firms, n_years = 500, 8
    n = n_firms * n_years
    firm_id  = np.repeat(np.arange(n_firms), n_years)
    year_id  = np.tile(np.arange(2016, 2016 + n_years), n_firms)

    # 时不变特征（跨年不变）
    soe_firm     = RNG.binomial(1, 0.35, n_firms)
    quality_firm = RNG.normal(0, 1, n_firms)           # 不可观测的公司质量
    size_base    = 1.5*soe_firm + 0.6*quality_firm + RNG.normal(0, 0.8, n_firms)

    soe       = np.repeat(soe_firm, n_years)
    quality   = np.repeat(quality_firm, n_years)
    Size_w    = np.repeat(size_base, n_years) + RNG.normal(0, 0.3, n)
    Leverage  = np.clip(
        np.repeat(RNG.beta(2,5,n_firms), n_years) - 0.08*soe + RNG.normal(0,0.05,n),
        0.05, 0.95)
    ROA_w     = RNG.normal(0.05, 0.03, n) - 0.02*soe + 0.03*quality

    year_shock = {y: v for y, v in zip(range(2016,2024),
                                        RNG.normal(0, 0.2, n_years))}
    eps  = RNG.normal(0, 0.3, n)
    # 真实系数：soe=-0.5, size=-0.3, lev=+0.8, roa=-0.5, quality=-0.4
    rate = (5.0 - 0.5*soe - 0.3*Size_w + 0.8*Leverage - 0.5*ROA_w
            - 0.4*quality + eps
            + np.array([year_shock[y] for y in year_id]))

    df = pd.DataFrame({
        'stkcd': firm_id.astype(str).str.zfill(6),
        'year' : year_id,
        'soe'  : soe.astype(int),
        'quality': quality,
        'Size_w' : Size_w,
        'Leverage_w': Leverage,
        'ROA_w'  : ROA_w,
        'avg_loan_rate': rate,
    })
    DATA_SOURCE = '模拟数据（真实系数：soe=-0.5, size=-0.3, lev=+0.8, roa=-0.5）'
    print(f'模拟数据：{df.shape}')

print(f'样本：{df["stkcd"].nunique()} 家公司，{df["year"].nunique()} 年')
print(f'来源：{DATA_SOURCE}')


---

## 2　从辛普森悖论理解固定效应

### 2.1　一个让人困惑的现象

考虑这个问题：公司规模（`Size_w`）和借款利率之间是什么关系？

- **截面角度**：规模大的公司，借款利率是否更低？
  → 跨公司比较，看不同规模公司的平均利率
- **时序角度**：当一家公司规模扩大时，它的借款利率如何变化？
  → 追踪同一家公司，看规模变化时利率如何响应

这两个问题的答案，可能方向完全相反。
下面的图形直观展示了这种现象——这正是 `simufe` 命令
（连玉君等，2022）所演示的核心直觉。


In [ ]:
# ── 2.2  simufe 风格模拟图：截面斜率 vs 组内斜率 ───────────────────────
# 复现 simufe 命令的核心图形逻辑：
# 同一张图里，截面 OLS 斜率（蓝色虚线）和组内 OLS 斜率（橙色实线）方向相反

# ── 构造一个「辛普森悖论」场景 ───────────────────────────────────────────
# 设定：公司质量（quality）是遗漏变量
#   quality → size（质量好的公司规模大）
#   quality → rate（质量好的公司利率低）
#   size    → rate（规模扩大时利率上升，因为借款多风险高）
# 结果：截面看 size 大→利率低（因为 quality 混淆），
#       但控制公司 FE 后，size 增大→利率上升（真实效应）

n_firms_demo = 6      # 只用 6 家公司，方便可视化
n_years_demo = 5
np.random.seed(2024)

# 每家公司的「质量」（时不变遗漏变量）
quality_demo = np.array([-1.5, -0.8, -0.2, 0.3, 1.0, 1.8])
colors_firm  = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd','#8c564b']

all_x, all_y, all_firm = [], [], []
firm_data = []

for i, (q, col) in enumerate(zip(quality_demo, colors_firm)):
    # 时变的规模：围绕公司均值（受 quality 影响）波动
    size_mean = 3.0 + 1.2 * q
    size_ts   = size_mean + np.linspace(-0.6, 0.6, n_years_demo)
    # 利率：quality 拉低利率，size 增大推高利率（真实效应 +0.4）
    rate_ts   = 4.5 - 0.8*q + 0.4*(size_ts - size_mean) + np.random.normal(0, 0.15, n_years_demo)
    firm_data.append({'size': size_ts, 'rate': rate_ts, 'color': col,
                       'label': f'公司 {i+1}（quality={q:.1f}）'})
    all_x.extend(size_ts)
    all_y.extend(rate_ts)
    all_firm.extend([i]*n_years_demo)

all_x = np.array(all_x)
all_y = np.array(all_y)

# ── 截面 OLS 斜率（忽略公司 FE）────────────────────────────────────────
slope_pooled, intercept_pooled, *_ = stats.linregress(all_x, all_y)

# ── 组内（Within）OLS 斜率 ─────────────────────────────────────────────
# 对每个变量做组内去均值
df_demo = pd.DataFrame({'x': all_x, 'y': all_y, 'firm': all_firm})
df_demo['x_dem'] = df_demo['x'] - df_demo.groupby('firm')['x'].transform('mean')
df_demo['y_dem'] = df_demo['y'] - df_demo.groupby('firm')['y'].transform('mean')
slope_within, intercept_within, *_ = stats.linregress(
    df_demo['x_dem'], df_demo['y_dem'])

print(f'截面 OLS 斜率（Pooled OLS）   ：{slope_pooled:+.4f}')
print(f'组内 OLS 斜率（Within / FE）  ：{slope_within:+.4f}')
print(f'真实 DGP 中 size 的系数（+0.4）：+0.4000')
print()
print('结论：截面斜率为负（规模大→利率低），但这是遗漏变量（quality）的影响。')
print('      组内斜率为正（当一家公司规模扩大时，利率上升），恢复了真实效应。')


In [ ]:
# ── 2.3  绘制 simufe 风格图形 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x_line = np.linspace(all_x.min()-0.2, all_x.max()+0.2, 100)

for ax_idx, ax in enumerate(axes):
    # ── 各公司的散点和连线 ──────────────────────────────────────────────
    for fd in firm_data:
        ax.plot(fd['size'], fd['rate'], 'o-',
                color=fd['color'], alpha=0.7, markersize=6,
                linewidth=1.5,
                label=fd['label'] if ax_idx == 0 else None)
        # 用灰色箭头连接同一公司相邻年份（展示时序轨迹）
        for t in range(len(fd['size'])-1):
            ax.annotate('',
                xy=(fd['size'][t+1], fd['rate'][t+1]),
                xytext=(fd['size'][t], fd['rate'][t]),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8, alpha=0.4))

    if ax_idx == 0:
        # ── 左图：截面 OLS（忽略公司 FE）────────────────────────────────
        y_line = slope_pooled * x_line + intercept_pooled
        ax.plot(x_line, y_line, 'b--', linewidth=2.5,
                label=f'Pooled OLS（截面斜率={slope_pooled:+.3f}）')
        ax.set_title('Pooled OLS：截面斜率（蓝色虚线）\n'
                     '规模大的公司利率更低——但这是遗漏变量的影响！',
                     fontsize=10)
    else:
        # ── 右图：组内 OLS（控制公司 FE）────────────────────────────────
        # 每家公司的组内拟合线（各自的去均值拟合）
        for fd in firm_data:
            x_m  = fd['size'].mean()
            y_m  = fd['rate'].mean()
            x_fe = np.linspace(fd['size'].min(), fd['size'].max(), 50)
            y_fe = y_m + slope_within * (x_fe - x_m)
            ax.plot(x_fe, y_fe, '-', color=fd['color'], linewidth=2,
                    alpha=0.9)
        # 整体 within 斜率示意线（从中心画）
        x_c = all_x.mean()
        y_c = all_y.mean()
        x_range = np.linspace(x_c - 1.2, x_c + 1.2, 50)
        ax.plot(x_range, y_c + slope_within*(x_range - x_c),
                'darkorange', linewidth=3, linestyle='-',
                label=f'Within / FE（组内斜率={slope_within:+.3f}）')
        ax.set_title('固定效应（FE）：组内斜率（橙色实线）\n'
                     '同一公司规模扩大时，利率上升——恢复了真实效应',
                     fontsize=10)

    ax.set_xlabel('公司规模（Size）', fontsize=10)
    ax.set_ylabel('借款利率（%）', fontsize=10)

    # 图例
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=8, loc='upper right')

plt.suptitle('辛普森悖论：截面斜率 vs 组内斜率',
             fontsize=12, y=1.01, fontweight='normal')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch06_simufe_style.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.4　这张图说明了什么

**左图（Pooled OLS）**：把所有公司、所有年份的数据放在一起做回归，
得到截面斜率为负——规模大的公司利率低。
但这不是「规模扩大导致利率降低」，
而是「质量好的公司恰好规模大、利率也低」——遗漏了 `quality` 这个混淆变量。

**右图（固定效应）**：每家公司的轨迹都向右上方倾斜——
当一家公司的规模扩大时，它的利率上升。
这才是 `size` 对 `rate` 的真实效应（DGP 中为 +0.4）。
固定效应通过**组内去均值**，消除了公司间差异（`quality` 等时不变特征），
只保留了每家公司内部的时序变动，得到正确方向的系数。

::: {.callout-important}
## 固定效应的本质：只用 Within Variation

个体固定效应（Entity FE）通过组内去均值，
**完全丢弃了截面信息（between variation）**，
只利用每个个体自身随时间的变化（within variation）来识别系数。

这意味着：
- **优点**：控制了所有时不变的个体特征，无论是否可观测
- **代价**：时不变的解释变量（如 `soe`）的系数无法识别，被 FE 吸收
:::

### 2.5　Stata 的 `simufe` 命令

上面的图形逻辑来自连玉君等（2022）开发的 Stata 外部命令 `simufe`，
它能生成交互式动画，展示随着固定效应数量增加，
系数估计如何从截面值逐步收敛到组内值。

```stata
* ── 安装并运行 simufe ────────────────────────────────────────────────
* ssc install simufe, replace   （或从 lianxh.cn 下载）
simufe, n(6) t(5) beta(0.4) quality    // 生成与上图对应的动态演示
```

参数说明：`n(6)` = 6 家公司，`t(5)` = 5 个时期，`beta(0.4)` = 真实 Within 系数。
运行后可以拖动滑块，直观看到 Pooled OLS 和 FE 系数的差异来源。


---

## 3　固定效应 = FWL 的特例

### 3.1　FWL 视角下的固定效应

第 5 章的 FWL 定理说：多元回归中某变量的系数，
等于将该变量和结果变量分别对其他变量残差化后做双变量回归的斜率。

加入公司固定效应虚拟变量 $d_i$，回归为：

$$Y_{it} = \alpha_i + X_{it}'\beta + \varepsilon_{it}$$

由 FWL 定理，$\hat{\beta}$ 等于先对 $Y_{it}$ 和 $X_{it}$
分别对虚拟变量组做**组内去均值（within demeaning）**，再做 OLS：

$$\tilde{Y}_{it} = Y_{it} - \bar{Y}_i, \quad
\tilde{X}_{it} = X_{it} - \bar{X}_i$$

双向固定效应则需要迭代去均值——先按公司去均值，再按年份去均值，
循环直至收敛（`pyfixest` 和 `reghdfe` 内部都用这个算法）。


In [ ]:
# ── 3.2  Python + FWL 验证 ──────────────────────────────────────────────
controls = [c for c in ['Size_w','Leverage_w','ROA_w'] if c in df.columns]
ctrl_str  = ' + '.join(controls)

# pyfixest：公司 FE（soe 被吸收）
m_fe = pf.feols(f'avg_loan_rate ~ {ctrl_str} | stkcd',
                 data=df, vcov={'CRV1':'stkcd'})

# 手动 FWL 验证：组内去均值
df_dem = df.copy()
for col in ['avg_loan_rate'] + controls:
    mean_i = df_dem.groupby('stkcd')[col].transform('mean')
    df_dem[f'{col}_w'] = df_dem[col] - mean_i

ctrl_w  = ' + '.join([f'{c}_w' for c in controls])
m_manual = smf.ols(f'avg_loan_rate_w ~ {ctrl_w} - 1', data=df_dem).fit()

print('FWL 验证（公司 FE = 组内去均值）：')
print(f"{'变量':<14} {'pyfixest FE':>14} {'手动去均值':>14} {'差值':>12}")
print('─' * 56)
for col in controls:
    c_pf = m_fe.coef().get(col, float('nan'))
    c_m  = m_manual.params.get(f'{col}_w', float('nan'))
    print(f'{col:<14} {c_pf:>14.6f} {c_m:>14.6f} {abs(c_pf-c_m):>12.2e}')
print('\n两种方法完全一致，FE = FWL 特例 ✓')


In [ ]:
# ── 3.3  soe 为什么被公司 FE 吸收 ──────────────────────────────────────
# soe 是时不变变量：对同一家公司，soe 在所有年份都相同
# 组内去均值后：soe_w = soe - mean(soe for firm) = soe - soe = 0

df_dem['soe_w'] = df['soe'] - df.groupby('stkcd')['soe'].transform('mean')
var_soe_w = df_dem['soe_w'].var()
var_size_w = df_dem['Size_w_w'].var() if 'Size_w_w' in df_dem.columns else None

print('去均值后各变量的方差：')
print(f'  soe_w   的方差：{var_soe_w:.6f}  （趋近于 0 → 无法识别系数）')
if var_size_w:
    print(f'  Size_w_w 的方差：{var_size_w:.6f}  （有变动 → 可以识别系数）')
print()
print('直觉：soe 是国企身份，同一家公司在样本期内几乎不变。')
print('      公司 FE 把这个「时不变」特征完全吸收，没有剩余变动来识别系数。')
print('      要估计 soe 的效应，需要利用「身份变化的公司」（DID）或工具变量。')


### 3.4　Stata 实现

```stata
%%stata
* ── 公司固定效应（xtreg fe）────────────────────────────────────────────
use "data_temp/for_stata.dta", clear
xtset stkcd year

* 公司 FE（注意：soe 等时不变变量会被自动删除）
xtreg avg_loan_rate Size_w Leverage_w ROA_w, fe vce(cluster stkcd)

* 双向 FE（reghdfe，推荐）
reghdfe avg_loan_rate Size_w Leverage_w ROA_w, ///
    absorb(stkcd year) vce(cluster stkcd)
est store twfe
```

**`pyfixest` 与 `reghdfe` 结果精确一致。**


---

## 4　高维固定效应（HDFE）

### 4.1　为什么需要 HDFE

有时研究设计需要同时控制多个维度的固定效应：

- **公司 FE + 年份 FE**（双向固定效应，最常见）
- **公司 FE + 行业 × 年份 FE**（控制行业-年份共同冲击）
- **省份 FE + 行业 FE + 年份 FE**（三维固定效应）

直接加虚拟变量会产生数万个参数，计算量爆炸。
HDFE 用**迭代去均值算法（Alternating Projections）**代替虚拟变量，
计算复杂度大幅下降，仍然基于 FWL 原理。

::: {.callout-warning}
## 过度控制导致解释变量被吸收

FE 维度越多，时不变或变化缓慢的解释变量越容易被吸收。

判断原则：如果某解释变量在某个 FE 维度内**几乎不变**，
加入该维度 FE 后这个变量的系数会消失或标准误极大。
这是信号——说明该 FE 和这个变量「共线」，需要取舍。
:::


In [ ]:
# ── 4.2  Python：HDFE 实现（pyfixest）────────────────────────────────────

# 添加行业变量（演示用）
if 'industry' not in df.columns:
    firm_to_ind = dict(zip(df['stkcd'].unique(),
                           RNG.integers(0, 10, df['stkcd'].nunique())))
    df['industry'] = df['stkcd'].map(firm_to_ind)

model_specs = {
    '(1) 年份FE'          : f'avg_loan_rate ~ soe + {ctrl_str} | year',
    '(2) 公司+年份（TWFE）': f'avg_loan_rate ~ {ctrl_str} | stkcd + year',
    '(3) 公司+行业×年份'  : f'avg_loan_rate ~ {ctrl_str} | stkcd + industry^year',
}

fitted = {}
for label, formula in model_specs.items():
    try:
        fitted[label] = pf.feols(formula, data=df, vcov={'CRV1':'stkcd'})
        print(f'{label}: N={fitted[label].nobs:,}  ✓')
    except Exception as e:
        print(f'{label}: 跳过（{e}）')

models_soe = [m for m in fitted.values() if 'soe' in m.coef().index]
if models_soe:
    pf.etable(models_soe, coef_fmt='b\n(se)', digits=3, stars=True)


In [ ]:
# ── 4.3  系数图：不同 FE 设置下 Size_w 系数的变化 ──────────────────────
coef_rows = []
for label, m in fitted.items():
    if 'Size_w' in m.coef().index:
        c  = m.coef()['Size_w']
        se = m.se()['Size_w']
        coef_rows.append({'模型': label, 'coef': c,
                           'lo': c-1.96*se, 'hi': c+1.96*se})

if coef_rows:
    cdf = pd.DataFrame(coef_rows)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    y_pos = range(len(cdf))
    ax.errorbar(cdf['coef'], y_pos,
               xerr=[cdf['coef']-cdf['lo'], cdf['hi']-cdf['coef']],
               fmt='o', color='steelblue', capsize=5, markersize=8)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    if DATA_SOURCE.startswith('模拟'):
        ax.axvline(-0.3, color='red', linewidth=1.2, linestyle=':',
                   label='真实系数 -0.3')
        ax.legend(fontsize=9)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(cdf['模型'], fontsize=9)
    ax.set_xlabel('Size_w 系数（点估计 + 95% CI）', fontsize=10)
    ax.set_title('不同 FE 设置下的 Size_w 系数', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch06_hdfe_coef.png', dpi=150, bbox_inches='tight')
    plt.show()


### 4.4　Stata 实现

```stata
%%stata
* ── HDFE（reghdfe）────────────────────────────────────────────────────
use "data_temp/for_stata.dta", clear

* 双向 FE
reghdfe avg_loan_rate soe Size_w Leverage_w ROA_w, ///
    absorb(stkcd year) vce(cluster stkcd)
est store twfe

* 三维 FE：公司 + 行业×年份
reghdfe avg_loan_rate Size_w Leverage_w ROA_w, ///
    absorb(stkcd industry#year) vce(cluster stkcd)
est store hdfe3

esttab twfe hdfe3, star(* 0.1 ** 0.05 *** 0.01) se b(%9.4f) r2 scalars(N)
```


---

## 5　Fama-MacBeth 回归

### 5.1　方法逻辑

**Fama-MacBeth（FM）回归**是资产定价实证研究的标准方法（Fama & MacBeth, 1973）。

**两步法：**

**第一步：每期截面回归**
在每一期 $t$，对截面数据做回归，得到该期的系数估计：

$$R_{it} = \lambda_{0t} + \lambda_{1t} \text{Size}_{it-1} + \cdots + \varepsilon_{it}$$

**第二步：时序均值推断**
对各期系数取时序均值，用时序标准差构造 $t$ 统计量：

$$\bar{\lambda}_1 = \frac{1}{T}\sum_t \hat{\lambda}_{1t}, \quad
t = \frac{\bar{\lambda}_1}{\text{SD}(\hat{\lambda}_{1t}) / \sqrt{T}}$$

标准误通常用 **Newey-West** 调整，以处理系数时序上的自相关。

### 5.2　FM vs TWFE 的适用场景

| 维度 | Fama-MacBeth | 面板 TWFE |
|------|-------------|-----------|
| 主要目的 | 资产定价因子检验 | 因果效应估计 |
| 标准误来源 | 系数的时序波动 | 聚类标准误 |
| 是否控制个体 FE | 否 | 是 |
| 时序相关 | 通过 Newey-West 处理 | 通过聚类处理 |
| 适用场景 | 「因子是否被定价」| 「控制固定效应后的偏效应」|


In [ ]:
# ── 5.3  Python 实现 ─────────────────────────────────────────────────────

def fama_macbeth(df, y, xs, time='year', nw_lags=1):
    '''Fama-MacBeth 两步法。返回均值系数、NW 标准误、t 值、p 值。'''
    coefs_t = []
    for t, gdf in df.groupby(time):
        gdf = gdf[[y]+xs].dropna()
        if len(gdf) < len(xs)+2:
            continue
        coefs_t.append(smf.ols(f'{y} ~ ' + ' + '.join(xs), data=gdf).fit().params)
    cdf = pd.DataFrame(coefs_t)
    T   = len(cdf)
    rows = []
    for var in cdf.columns:
        lam  = cdf[var].values
        mean = lam.mean()
        # Newey-West 标准误
        nw_se = sm.OLS(lam, np.ones(T)).fit(
            cov_type='HAC', cov_kwds={'maxlags': nw_lags}).bse[0]
        t_val = mean / nw_se
        p_val = 2*(1 - stats.t.cdf(abs(t_val), df=T-1))
        rows.append({'变量': var, '均值λ': mean, 'NW_SE': nw_se,
                     't值': t_val, 'p值': p_val})
    return pd.DataFrame(rows).set_index('变量')

xs_fm = [c for c in ['soe','Size_w','Leverage_w','ROA_w'] if c in df.columns]
fm_res = fama_macbeth(df, 'avg_loan_rate', xs_fm)

print('Fama-MacBeth 回归（Newey-West SE，lag=1）：')
print(fm_res.round(4).to_string())


In [ ]:
# ── 5.4  用 linearmodels 的接口（更简洁）────────────────────────────────
df_panel = df.set_index(['stkcd','year'])
fm_lm_res = FM_lm.from_formula(
    f'avg_loan_rate ~ soe + {ctrl_str}',
    data=df_panel
).fit(bandwidth=1)
print('linearmodels.FamaMacBeth 结果：')
print(fm_lm_res.summary.tables[1])


### 5.5　Stata 实现

```stata
%%stata
* ── Fama-MacBeth（xtfmb）──────────────────────────────────────────────
* 需要安装：ssc install xtfmb, replace
use "data_temp/for_stata.dta", clear
xtset stkcd year

xtfmb avg_loan_rate soe Size_w Leverage_w ROA_w, lag(1)
```

**`linearmodels.FamaMacBeth` 的均值系数和 t 值应与 Stata `xtfmb` 一致。**


---

## 6　Portfolio Sort 分组分析

### 6.1　什么是 Portfolio Sort

Portfolio Sort 是 FM 回归的**非参数替代**：
不假设因子与收益的关系是线性的，
直接比较极端分组的均值差（spread）来检验因子效应。

**标准流程（单变量 Sort）：**
1. 每期按排序变量（如 `ROA_w`）从小到大分成 $Q$ 组（通常 $Q=5$ 或 $Q=10$）
2. 计算各组的结果变量（如借款利率）均值
3. 计算第 $Q$ 组与第 1 组的 **$Q$-1 Spread**
4. 对 spread 的时序序列做 $t$ 检验（是否显著异于 0）

### 6.2　双重分组（Double Sort）

控制第二个变量的影响：先按 $Z$ 分 $m$ 组，在每个 $Z$ 组内再按 $X$ 分 $k$ 组，
输出 $k \times m$ 的均值矩阵。
这是在不假设线性的前提下，检验「控制 $Z$ 后 $X$ 是否仍有效应」的标准工具。


In [ ]:
# ── 6.3  单变量 Portfolio Sort ───────────────────────────────────────────

def portfolio_sort(df, sort_var, ret_var, n_groups=5, time='year'):
    '''单变量分组分析，返回各组均值和 Q-1 spread 的 t 检验。'''
    group_rets = {g: [] for g in range(1, n_groups+1)}
    spreads    = []
    for t, gdf in df.groupby(time):
        gdf = gdf[[sort_var, ret_var]].dropna()
        if len(gdf) < n_groups * 3:
            continue
        gdf = gdf.copy()
        gdf['grp'] = pd.qcut(gdf[sort_var], n_groups,
                              labels=range(1, n_groups+1))
        means = gdf.groupby('grp')[ret_var].mean()
        for g in range(1, n_groups+1):
            if g in means.index:
                group_rets[g].append(float(means[g]))
        if 1 in means.index and n_groups in means.index:
            spreads.append(float(means[n_groups] - means[1]))
    group_means = {g: np.mean(v) for g, v in group_rets.items() if v}
    spread_mean = np.mean(spreads) if spreads else float('nan')
    t_stat, p_val = (stats.ttest_1samp(spreads, 0)
                     if len(spreads) >= 3 else (float('nan'), float('nan')))
    return group_means, spread_mean, t_stat, p_val

if 'ROA_w' in df.columns:
    gm, spread, t, p = portfolio_sort(df, 'ROA_w', 'avg_loan_rate', n_groups=5)
    sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else '（不显著）'
    print('ROA_w 分组 → 平均借款利率：')
    print(f"  {'分组':<6} {'利率均值（%）':>12}")
    print('  ' + '─' * 20)
    for g, m in sorted(gm.items()):
        tag = '（最低ROA）' if g==1 else '（最高ROA）' if g==5 else ''
        print(f'  Q{g}{tag:<12} {m:>8.4f}')
    print(f"  {'Q5-Q1 Spread':<16} {spread:>6.4f}%   t={t:.3f}  p={p:.4f} {sig}")


In [ ]:
# ── 6.4  可视化：分组均值图 ─────────────────────────────────────────────
if 'ROA_w' in df.columns:
    gm_df = pd.DataFrame({'grp': list(gm.keys()), 'mean': list(gm.values())})
    gm_df['label'] = gm_df['grp'].map(
        {1:'Q1\n（最低）',2:'Q2',3:'Q3',4:'Q4',5:'Q5\n（最高）'})

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(gm_df['label'], gm_df['mean'],
           color=['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd'],
           edgecolor='white', width=0.6)
    ax.axhline(df['avg_loan_rate'].mean(), color='gray', linestyle='--',
               linewidth=1, label='全样本均值')
    ax.set_xlabel('ROA 分组（Q1=最低盈利能力）', fontsize=10)
    ax.set_ylabel('平均借款利率（%）', fontsize=10)
    ax.set_title('Portfolio Sort：ROA 分组与借款利率', fontsize=11)
    ax.legend(fontsize=9)
    txt = f'Q5-Q1 = {spread:.3f}%\nt={t:.2f}, p={p:.3f}'
    ax.annotate(txt, xy=(0.97,0.05), xycoords='axes fraction',
               ha='right', va='bottom', fontsize=9,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch06_portfolio_sort.png', dpi=150, bbox_inches='tight')
    plt.show()


::: {.callout-tip}
## 提示词模板：FM + Portfolio Sort 完整检验

````
我有面板 DataFrame df，列为：stkcd, year, ret（下期收益率），
以及特征变量 Size, BM, ROA。

请完成资产定价截面检验：

第一步：单变量 Portfolio Sort
- 按 ROA 每期分成 5 组（Quintile）
- 计算各组下期平均收益率，绘制分组均值柱状图
- 对 Q5-Q1 spread 做时序 t 检验（Newey-West，lag=1）

第二步：Fama-MacBeth 回归
- 每期截面：ret ~ Size + BM + ROA
- 对各期系数取均值，用 Newey-West 标准误（lag=1）做 t 检验
- 输出均值、NW_SE、t 值、p 值

第三步：Double Sort
- 先按 BM 分 3 组，在每个 BM 组内再按 ROA 分 3 组
- 输出 3×3 的平均收益率矩阵，行/列都要加 spread 行/列
````
:::


---

## 7　综合案例：面板分析全流程


In [ ]:
# ── 7.1  从 Pooled OLS 到 TWFE：观察系数如何变化 ────────────────────────
print('=' * 65)
print(f'来源：{DATA_SOURCE}')
print(f'样本：{df["stkcd"].nunique()} 家公司  {df["year"].nunique()} 年  N={len(df)}')
print('=' * 65)

base_ctrl = [c for c in ['Size_w','Leverage_w','ROA_w'] if c in df.columns]
cf = ' + '.join(base_ctrl)

r1 = pf.feols(f'avg_loan_rate ~ soe + {cf}',
               data=df, vcov={'CRV1':'stkcd'})
r2 = pf.feols(f'avg_loan_rate ~ soe + {cf} | year',
               data=df, vcov={'CRV1':'stkcd'})
r3 = pf.feols(f'avg_loan_rate ~ {cf} | stkcd',
               data=df, vcov={'CRV1':'stkcd'})
r4 = pf.feols(f'avg_loan_rate ~ {cf} | stkcd + year',
               data=df, vcov={'CRV1':'stkcd'})

pf.etable([r1,r2,r3,r4], coef_fmt='b\n(se)', digits=3, stars=True,
          labels={'avg_loan_rate':'借款利率（%）','soe':'国企（=1）',
                  'Size_w':'规模','Leverage_w':'杠杆率','ROA_w':'ROA'})


In [ ]:
# ── 7.2  结论输出 ────────────────────────────────────────────────────────
lev_ols  = r1.coef()['Leverage_w']
lev_twfe = r4.coef()['Leverage_w']

print('── 关键结论 ────────────────────────────────────────────────────')
print(f'Leverage_w 系数：Pooled OLS = {lev_ols:.4f}，TWFE = {lev_twfe:.4f}')
if DATA_SOURCE.startswith('模拟'):
    print(f'真实系数（DGP）= +0.8，TWFE 估计误差 = {lev_twfe - 0.8:.4f}')
print()
print('从 Pooled OLS 到 TWFE 的系数变化，说明：')
print('1. Pooled OLS 混入了公司间差异（遗漏变量 quality）和宏观年份效应。')
print('2. 加入公司 FE 后，soe 被吸收（时不变变量无法识别）——')
print('   这不是说国企效应不存在，而是说公司 FE 无法识别时不变处理变量。')
print('3. TWFE 给出更干净的偏效应，但要做因果推断仍需更强的识别策略（第 8 章）。')


---

## 8　章末练习

**练习 1（辛普森悖论）**
用本章的 `df` 数据，找一对变量，它们的**截面**相关方向
与**组内（within）**相关方向相反。
绘制 simufe 风格的散点图（每家公司用不同颜色，加 Pooled 和 Within 拟合线），
并用一段话解释为什么会出现这种情况。

**练习 2（FWL 验证）**
对双向 FE（公司 + 年份）模型，手动执行迭代双向去均值（至少循环 20 次），
验证收敛后的系数与 `pyfixest` TWFE 一致。
绘制「迭代次数 vs 系数收敛过程」折线图。

**练习 3（FM + Sort）**
用 `Leverage_w` 作为排序变量，完成：
① 单变量 Portfolio Sort（5 组），报告 Q5-Q1 spread 和 $t$ 值
② FM 回归，比较 `Leverage_w` 系数与 Portfolio Sort spread 的方向是否一致
③ 简要说明两种方法结论差异的可能原因

**练习 4（思考题）**
加入公司 FE 后，`soe` 的系数消失了。
若有研究者声称：「发现某家民营企业在 2020 年被国有化，
其之后的借款利率显著下降」，
这种研究设计（利用国企化事件）能否识别 `soe` 的因果效应？
需要什么额外假设？这与 DID 的逻辑有什么关系？
